In [1]:
from loguru import logger
from plant_care.config import get_env, load_config
from plant_care.data_ingestor import DataIngestor
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.getOrCreate()

# Load config
env = get_env()
cfg = load_config("../project_config.yml", env)

CATALOG = cfg.catalog
SCHEMA = cfg.schema
TABLE_NAME = "plant_care"

# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
logger.info(f"Schema {CATALOG}.{SCHEMA} ready")

2026-11-17 14:22:38.784 | INFO     | __main__:<module>:13 - Schema llmops_dev.srinidhi_ilango98 ready


In [3]:
# Fetch plant care data
logger.info("Fetching plant care data...")
ingestor = DataIngestor(spark, cfg)
plants = ingestor.fetch_plant_care()
logger.info(f"Fetched {len(plants)} plant profiles")
if plants:
    logger.info(f"Sample plant: {plants[0]['common_name']} ({plants[0]['scientific_name']})")

2026-11-17 14:22:38.799 | INFO     | __main__:<module>:2 - Fetching plant care data...
2026-11-17 14:22:39.118 | INFO     | plant_care.data_ingestor:fetch_plant_care:208 - Found 143 plants in PlantSolve index
2026-11-17 14:23:06.620 | INFO     | __main__:<module>:5 - Fetched 143 plant profiles
2026-11-17 14:23:06.620 | INFO     | __main__:<module>:7 - Sample plant: African Violet (Streptocarpus sect. Saintpaulia)


In [4]:
# Create Delta Table in Unity Catalog
# Store the plant care metadata in a Delta table for downstream processing.
df = ingestor.create_dataframe(plants)
logger.info(f"Created DataFrame with {df.count()} plant records")

2026-11-17 14:23:16.628 | INFO     | __main__:<module>:4 - Created DataFrame with 143 plant records


In [5]:
# Write to Delta table
table_path = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_path)

logger.info(f"Created Delta table: {table_path}")
logger.info(f"Records: {df.count()}")

2026-11-17 14:23:38.003 | INFO     | __main__:<module>:6 - Created Delta table: llmops_dev.srinidhi_ilango98.plant_care
2026-11-17 14:23:38.475 | INFO     | __main__:<module>:7 - Records: 143


In [7]:
# COMMAND ----------
# Verify the Data

# Read back the table
plants_df = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")

logger.info(f"Table: {CATALOG}.{SCHEMA}.{TABLE_NAME}")
logger.info(f"Total plants: {plants_df.count()}")
logger.info("Schema:")
plants_df.printSchema()

logger.info("Sample records:")
plants_df.select("plant_id", "slug", "common_name", "scientific_name", "volume_path").show(5, truncate=50)

# COMMAND ----------
# Data Statistics

logger.info("Plants by category:")
plants_df.groupBy("category").count().orderBy("count", ascending=False).show()


2026-11-17 14:24:04.367 | INFO     | __main__:<module>:6 - Table: llmops_dev.srinidhi_ilango98.plant_care
2026-11-17 14:24:05.172 | INFO     | __main__:<module>:7 - Total plants: 143
2026-11-17 14:24:05.172 | INFO     | __main__:<module>:8 - Schema:
2026-11-17 14:24:05.504 | INFO     | __main__:<module>:11 - Sample records:


root
 |-- plant_id: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- common_name: string (nullable = true)
 |-- scientific_name: string (nullable = true)
 |-- family: string (nullable = true)
 |-- category: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- difficulty: string (nullable = true)
 |-- placement: string (nullable = true)
 |-- watering: string (nullable = true)
 |-- soil: string (nullable = true)
 |-- toxicity: string (nullable = true)
 |-- light_description: string (nullable = true)
 |-- sunlight_hours: string (nullable = true)
 |-- light_intensity: string (nullable = true)
 |-- temperature_recommended: string (nullable = true)
 |-- humidity_recommended: string (nullable = true)
 |-- soil_ph_recommended: string (nullable = true)
 |-- fertilizer_frequency: string (nullable = true)
 |-- growth_rate: string (nullable = true)
 |-- mature_height: string (nullable = true)
 |-- pet_safe_cat: boolean (nullable 

2026-11-17 14:24:06.089 | INFO     | __main__:<module>:16 - Plants by category:


+---------------------------+---------------------+---------------------+-----------------------------------------------+--------------------------------------------------+
|                   plant_id|                 slug|          common_name|                                scientific_name|                                       volume_path|
+---------------------------+---------------------+---------------------+-----------------------------------------------+--------------------------------------------------+
|       plant-african-violet|       african-violet|       African Violet|                Streptocarpus sect. Saintpaulia|/Volumes/llmops_dev/srinidhi_ilango98/plant_car...|
|   plant-aglaonema-red-siam|   aglaonema-red-siam|   Aglaonema Red Siam|                        Aglaonema 'Siam Aurora'|/Volumes/llmops_dev/srinidhi_ilango98/plant_car...|
|   plant-alocasia-amazonica|   alocasia-amazonica|   Alocasia Amazonica|Alocasia x amazonica (commonly sold as 'Polly')|/Volumes/llmop

In [8]:
# Review one sample article
sample = plants_df.filter("volume_path IS NOT NULL").select("slug", "volume_path").first()
logger.info(f"Sample article: {sample['slug']} ({sample['volume_path']})")

with open(sample["volume_path"], encoding="utf-8") as f:
    html = f.read()
logger.info(f"Article length: {len(html)} characters")
print(html[:2000])

2026-11-17 14:25:30.982 | INFO     | __main__:<module>:3 - Sample article: african-violet (/Volumes/llmops_dev/srinidhi_ilango98/plant_care/articles/african-violet.html)
2026-11-17 14:25:31.015 | INFO     | __main__:<module>:7 - Article length: 5753 characters


<h2>Overview</h2><p>The African Violet (Saintpaulia) is one of the most beloved indoor flowering plants in the world, prized for its ability to produce vibrant clusters of purple, pink, or white flowers year-round. However, they also have a reputation for being finicky, largely because they are highly susceptible to crown rot and leaf spotting if watered incorrectly. The key to mastering African Violet care is understanding that their fuzzy leaves are a delicate defense mechanism that cannot get wet. This guide will teach you the art of bottom-watering and how to find the perfect light balance to keep your plant blooming continuously.</p><p>Native to the humid cloud forests of Tanzania and Kenya, African Violets grow in rocky crevices or on the forest floor, sheltered from direct sun and heavy downpours. They absorb moisture from the humid air and the damp, mossy ground. Their leaves are covered in tiny hairs (trichomes) designed to collect dew. When we water them from above with cold 

In [3]:
# Some do not have articles so no volume_path
spark.table("llmops_dev.srinidhi_ilango98.plant_care").filter("volume_path IS NULL").select("slug", "common_name").show(25, truncate=False)

+--------------------------+--------------------------+
|slug                      |common_name               |
+--------------------------+--------------------------+
|cactus                    |cactus                    |
|cymbidium-orchid          |Cymbidium Orchid          |
|cyperus-papyrus           |Papyrus                   |
|dendrobium-orchid         |Dendrobium Orchid         |
|echinocactus-grusonii     |Golden Barrel Cactus      |
|ficus-elastica            |Rubber Plant              |
|ficus-lyrata              |Fiddle Leaf Fig           |
|heartleaf-philodendron    |Heartleaf Philodendron    |
|kentia-palm               |Kentia Palm               |
|lemon-tree                |Lemon Tree                |
|maidenhair-fern           |Maidenhair Fern           |
|mini-monstera             |Mini Monstera             |
|monstera-adansonii        |Swiss Cheese Vine         |
|monstera-deliciosa        |Swiss Cheese Plant        |
|monstera                  |Monstera            